### Describing the farm

First of all we need a way to describe the farm according to the principles of Object Oriented Programming.

**General contact details for the farm**

```json
"farm_info": {
    "farm_name": "AP_Farm",
    "farmer": "Augustin",
    "id": "unique_id"
}
```

Then we describe each field of the farm. We do not want o tie the fields to a specific crop, becasue this can change over time.

**Fields**

```json
{
    "id": "AP_Farm_field_1",
    "alias": "Parcela Nord",
    "area_ha": 5.0,
    "boundary": [
        {
            "lat": 46.83272043023766,
            "lon": 24.17629055822008
        },
        {
            "lat": 46.83318284358206,
            "lon": 24.176730440479606
        },
        {
            "lat": 46.83320486316589,
            "lon": 24.177159593903536
        },
        {
            "lat": 46.83290392807272,
            "lon": 24.177878425888625
        },
        {
            "lat": 46.832368112638086,
            "lon": 24.17746000130029
        }
    ]
}
```

Now the most important part is the crop description.

**Crops**

*General information*

```json
{
    "id": "AP_Farm_crop_1",
    "alias": "Lucernă",
    "field_id": "AP_Farm_field_1",
    "scientific_parameters":{}
}
```

*Scientific parameters*

Growing Degree Days (GDD) 
```json
    "gdd_params": {
        "base_temp_c": 5.0,
        "optimal_temp_c": 25.0,
        "max_temp_c": 35.0,
        "method": "standard"
    },
```

BBCH thresholds
```json
    "bbch_thresholds": {
        "09": 100,
        "10": 150,
        "50": 400,
        "60": 550
    },
```

Water parameters
```json
    "water_params": {
        "kc_initial": 0.4,
        "kc_mid": 0.95,
        "kc_end": 0.9,
        "rooting_depth_max_m": 2.0,
        "depletion_fraction_p": 0.5
    },
    "stress_thresholds": {
        "moisture_stress_index_limit": 0.5
    }
```


In [ ]:
# Step 0: Import clasees in the source directory

import sys
import os

# Add the src directory to the python path
sys.path.append(os.path.abspath('../src'))

# Now you can import your classes
from farm_model import FarmModel

# Load the model
farm = FarmModel.load("../data/farm_model.json")
print(f"Loaded farm: {farm.farm_info.name}")

In [ ]:
# Step 1: Download data from CDS

# Link to dataset: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview

import cdsapi

dataset = "reanalysis-era5-land-timeseries"
request = {
    "variable": [
        "2m_temperature"
        ],
    "location": {
        # TODO: get these coordinates from Farm object
        "longitude": 24.1, 
        "latitude": 46.8},
    "date": ["2025-04-01/2025-09-30"],
    "data_format": "netcdf",
    "download_format": "archived"
}

zip_file_path = "../download/temps.zip"
unzipped_folder_path = "../download/temps/"

client = cdsapi.Client()
client.retrieve(dataset, request, zip_file_path)

In [ ]:
# Step 2: Unzip the downloaded data
import zipfile

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(unzipped_folder_path)

In [ ]:
# Step 3: Read the all the data in the unzipped folder

import glob
import xarray as xr

# List all NetCDF files in the folder
files = glob.glob(unzipped_folder_path + "*.nc")

for file in files:
    print(f"Processing file: {file}")

    ds = xr.open_dataset(file)
    print(ds)

    df = ds.to_dataframe().reset_index()
    print(f"Number of columns: {df.shape[1]}")
    print(df.columns)       # Column names
    print(df.head())


In [ ]:
# Step 3.1: Start analysing a specific file (reanalysis-era5-land-timeseries-sfc-2m-temperatureami76x0c)

import pandas as pd
import matplotlib.pyplot as plt

nc_file_path = "../download/temps/reanalysis-era5-land-timeseries-sfc-2m-temperatureami76x0c.nc"

ds = xr.open_dataset(nc_file_path)

# Convert temperature from Kelvin to Celsius
temp_c = ds['t2m'] - 273.15
print(temp_c)

# Convert to DataFrame
df = temp_c.to_dataframe().reset_index()
print(df.head())

df['date'] = pd.to_datetime(df['valid_time']).dt.date
df['time'] = pd.to_datetime(df['valid_time']).dt.time
print(df.head())

df = df.rename(columns={'t2m': 'Temperature_C'})
print(df.head())

df_daily = df.groupby('date')['Temperature_C'].agg(['min', 'max', 'mean']).reset_index()
df_daily.rename(columns={'min': 'Tmin', 'max': 'Tmax', 'mean': 'Tavg'}, inplace=True)
print(df_daily.head())


#TODO: find a more elegant way to get this value per crop type
Tbase = 6  # °C, for lucerne (you can change to 8 for sunflower)
df_daily['GDD'] = df_daily.apply(lambda r: max(0, ((r['Tmax'] + r['Tmin'])/2) - Tbase), axis=1)

# --- Step 4: Cumulative GDD ---
df_daily['GDD_cumulative'] = df_daily['GDD'].cumsum()
df_daily = df_daily.set_index('date')
print(df_daily.head())




In [ ]:
# Example GDD thresholds for lucerne (adjust as needed)
bbch_thresholds_lucerne = {
    "00-09 Germination / emergence (seedling visible)": 50,   # small GDD after sowing to emerge
    "10-19 Leaf development / vegetative": 200,
    "30-39 Stem elongation": 400,
    "50-59 Flower buds (early bud)": 700,     # candidate for first cut (approx)
    "60-69 Flowering (full bloom)": 1000,
    "70-79 Fruit / seed set": 1300,
    "80-89 Ripening / senescence": 1600
}

# find first date crossing each threshold
start = pd.to_datetime(pd.Timestamp("2025-04-01")).date()

stage_dates = {}
for stage, thr in bbch_thresholds_lucerne.items():
    # find first index where GDD_cum >= thr and date >= planting_date
    reached = df_daily[(df_daily.index >= start) & (df_daily['GDD_cumulative'] >= thr)]
    if not reached.empty:
        stage_dates[stage] = pd.to_datetime(reached.index[0]).date()
    else:
        stage_dates[stage] = None

plt.figure(figsize=(12,6))
dates = [pd.to_datetime(d) for d in df_daily.index]
plt.plot(dates, df_daily['GDD_cumulative'], label='Cumulative GDD', color='tab:green', linewidth=2)
plt.fill_between(dates, 0, df_daily['GDD_cumulative'], color='tab:green', alpha=0.1)
plt.xlabel('Date'); plt.ylabel('Cumulative GDD (°C·day)')
plt.title("Cumulative Growing Degree Days (GDD)")
plt.grid(alpha=0.3)

# annotate stage dates
for stage, dt in stage_dates.items():
    if dt is None:
        continue
    dt_ts = pd.to_datetime(dt)
    gdd_val = df_daily.loc[dt, 'GDD_cumulative']
    plt.axvline(x=dt_ts, color='gray', linestyle='--', alpha=0.6)
    plt.text(dt_ts, gdd_val + max(df_daily['GDD_cumulative'])*0.03, f"{stage}\n{dt}", rotation=45,
            fontsize=9, ha='left', va='bottom', color='darkslategray')

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot
plt.figure(figsize=(12,6))
plt.plot(df['date'], df['Temperature_C'], label='2m Temperature (°C)')
plt.title("ERA5-Land Temperature at 2m (46.8N, 24.1E)")
plt.xlabel("Date")
plt.ylabel("°C")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df_daily['date'], df_daily['GDD_cumulative'], label='Cumulative GDD', color='green')
plt.title('Cumulative Growing Degree Days (ERA5, 2025)')
plt.xlabel('Date')
plt.ylabel('GDD (°C·day)')
plt.grid(True)
plt.legend()
plt.show()